In [23]:
import os
import operator
from typing import TypedDict, Annotated,Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import google.genai as genai
from google.genai import types
from langgraph.graph import StateGraph, START, END



In [24]:
class conditionState(TypedDict):
    a:int
    b:int
    c:int
    equation:str
    d:float
    result:float

In [25]:
def show_equation(state:conditionState):
    equation = f'{state["a"]}x2{state["b"]}x{state["c"]}'
    return {'equation':equation}

def cal_d(state:conditionState):
    d = state['b']**2- (4*state['a']*state['c'])
    return {'d':d}



In [26]:
# calculate the roots 
def real_roots(state: conditionState):

    root1 = (-state["b"] + state["d"]**0.5)/(2*state["a"])
    root2 = (-state["b"] - state["d"]**0.5)/(2*state["a"])

    result = f'The roots are {root1} and {root2}'

    return {'result': result}

def repeated_roots(state: conditionState):

    root = (-state["b"])/(2*state["a"])

    result = f'Only repeating root is {root}'

    return {'result': result}

def no_real_roots(state: conditionState):

    result = f'No real roots'

    return {'result': result}

def check_condition(state: conditionState) -> Literal["real_roots", "repeated_roots", "no_real_roots"]:

    if state['d'] > 0:
        return "real_roots"
    elif state['d'] == 0:
        return "repeated_roots"
    else:
        return "no_real_roots"

In [27]:
graph = StateGraph(conditionState)

# node creatio
graph.add_node('show_equation',show_equation)
graph.add_node('calculate_d',cal_d)
graph.add_node('no_real_roots',no_real_roots)
graph.add_node('real_roots',real_roots)
graph.add_node('repeated_roots',repeated_roots)

# add edge 
graph.add_edge(START,'show_equation')
graph.add_edge('show_equation','calculate_d')
graph.add_conditional_edges('calculate_d', check_condition)
graph.add_edge('real_roots', END)
graph.add_edge('repeated_roots', END)
graph.add_edge('no_real_roots', END)

In [28]:
workflow = graph.compile()

In [30]:
inital_state ={
    'a':2,
    'b':4,
    'c':2
}
workflow.invoke(inital_state)

{'a': 2,
 'b': 4,
 'c': 2,
 'equation': '2x24x2',
 'd': 0,
 'result': 'Only repeating root is -1.0'}